<a href="https://colab.research.google.com/github/Syed1611/curvature-tuning-research/blob/main/notebooks/masters_research_testing_on_curve_tunings_30_epochs_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q -r requirements.txt

!pip install numpy 1.26.3

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
ERROR: Could not find a version that satisfies the requirement 1.26.3 (from versions: none)
ERROR: No matching distribution found for 1.26.3


In [3]:
import torch
import numpy
import pandas
import sklearn
import datasets
import tqdm

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("scikit-learn:", sklearn.__version__)
print("datasets:", datasets.__version__)
print("tqdm:", tqdm.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.6.0+cu124
CUDA: True
NumPy: 1.26.3
Pandas: 2.2.3
scikit-learn: 1.5.2
datasets: 3.4.1
tqdm: 4.66.5
GPU: Tesla T4


In [4]:
import os
os.environ["WANDB_MODE"] = "disabled"

In [5]:
!nvidia-smi

Fri Sep 25 19:43:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
%cd /content
!git clone https://github.com/Leon-Leyang/curvature-tuning
%cd curvature-tuning
!git rev-parse HEAD
!git status

/content
fatal: destination path 'curvature-tuning' already exists and is not an empty directory.
/content/curvature-tuning
6313b55e34501130ed9e7051371ca5054bb426f8
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   utils/data.py

no changes added to commit (use "git add" and/or "git commit -a")


In [7]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


In [8]:
from datasets import load_dataset

data_files = {
    "train": "hf://datasets/AI-Lab-Makerere/beans/data/train-00000-of-00001.parquet",
    "validation": "hf://datasets/AI-Lab-Makerere/beans/data/validation-00000-of-00001.parquet",
    "test": "hf://datasets/AI-Lab-Makerere/beans/data/test-00000-of-00001.parquet",
}

beans = load_dataset("parquet", data_files=data_files)

print(beans)
print("Train:", len(beans["train"]))
print("Validation:", len(beans["validation"]))
print("Test:", len(beans["test"]))
print("Columns:", beans["train"].column_names)
print(beans["train"].features)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


data/train-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})
Train: 1034
Validation: 133
Test: 128
Columns: ['image_file_path', 'image', 'labels']
{'image_file_path': Value(dtype='string', id=None), 'image': Image(mode=None, decode=True, id=None), 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'], id=None)}


In [9]:
from pathlib import Path

path = Path("/content/curvature-tuning/utils/data.py")
text = path.read_text()

old = """    elif dataset_to_use in ['fgvc-aircraft','flowers102','beans','dtd','celeb-a']:
        hf_trainset = datasets.load_dataset(
            f"randall-lab/{dataset_to_use}",
            split="train",
            trust_remote_code=True
        )
        hf_valset = datasets.load_dataset(
            f"randall-lab/{dataset_to_use}",
            split="validation",
            trust_remote_code=True
        )
        hf_testset = datasets.load_dataset(
            f"randall-lab/{dataset_to_use}",
            split="test",
            trust_remote_code=True
        )
        train_set  = HuggingFaceDataset(hf_trainset, transform=transform_train)
        val_set    = HuggingFaceDataset(hf_valset, transform=transform_test)
        test_set    = HuggingFaceDataset(hf_testset, transform=transform_test)
"""

new = """    elif dataset_to_use == 'beans':
        data_files = {
            "train": "hf://datasets/AI-Lab-Makerere/beans/data/train-00000-of-00001.parquet",
            "validation": "hf://datasets/AI-Lab-Makerere/beans/data/validation-00000-of-00001.parquet",
            "test": "hf://datasets/AI-Lab-Makerere/beans/data/test-00000-of-00001.parquet",
        }

        beans_ds = datasets.load_dataset(
            "parquet",
            data_files=data_files
        )

        hf_trainset = beans_ds["train"]
        hf_valset = beans_ds["validation"]
        hf_testset = beans_ds["test"]

        # Current HF copy calls the class column "labels".
        # The authors' wrapper expects "label".
        hf_trainset = hf_trainset.rename_column("labels", "label")
        hf_valset = hf_valset.rename_column("labels", "label")
        hf_testset = hf_testset.rename_column("labels", "label")

        train_set = HuggingFaceDataset(
            hf_trainset, transform=transform_train
        )
        val_set = HuggingFaceDataset(
            hf_valset, transform=transform_test
        )
        test_set = HuggingFaceDataset(
            hf_testset, transform=transform_test
        )

    elif dataset_to_use in ['fgvc-aircraft','flowers102','dtd','celeb-a']:
        hf_trainset = datasets.load_dataset(
            f"randall-lab/{dataset_to_use}",
            split="train",
            trust_remote_code=True
        )
        hf_valset = datasets.load_dataset(
            f"randall-lab/{dataset_to_use}",
            split="validation",
            trust_remote_code=True
        )
        hf_testset = datasets.load_dataset(
            f"randall-lab/{dataset_to_use}",
            split="test",
            trust_remote_code=True
        )
        train_set  = HuggingFaceDataset(hf_trainset, transform=transform_train)
        val_set    = HuggingFaceDataset(hf_valset, transform=transform_test)
        test_set   = HuggingFaceDataset(hf_testset, transform=transform_test)
"""

if old not in text:
    raise RuntimeError("Original data-loading block not found. Stop here.")

text = text.replace(old, new)
path.write_text(text)

print("Beans compatibility patch applied correctly.")

RuntimeError: Original data-loading block not found. Stop here.

In [10]:
%cd /content/curvature-tuning

from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_beans",
    train_batch_size=32,
    test_batch_size=800,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

/content/curvature-tuning
Train samples: 1034
Validation samples: 133
Test samples: 128


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([1, 1, 0, 1, 0, 2, 1, 0, 1, 1])


In [11]:
from pathlib import Path

path = Path("train.py")
text = path.read_text()

old = "for epoch in range(1, 21):"
new = "for epoch in range(1, 31):"

count = text.count(old)

print("Occurrences found:", count)

if count != 1:
    raise RuntimeError(
        f"Expected exactly 1 matching linear-probe loop, found {count}"
    )

text = text.replace(old, new, 1)
path.write_text(text)

print("Changed linear probing from 20 epochs to 30 epochs.")

Occurrences found: 1
Changed linear probing from 20 epochs to 30 epochs.


In [12]:
!grep -n "for epoch in range" train.py

184:        for epoch in range(1, 31):
301:    for epoch in range(start_epoch, num_epochs + 1):


In [13]:
!python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800

2026-09-25 19:44:11.186 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed42.log
2026-09-25 19:44:11.189 | INFO     | __main__:main:58 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 131MB/s]
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
2026-09-25 19:44:12.978 | INFO     | __main__:main:85 - Testing baseline...
2026-09-25 19:44:12.993 | INFO     | __main__:main:88 - Number of trainable parameters: 1539
20

In [14]:
!cat results/base_imagenet_to_beans_resnet18_seed42.json
!cat results/ct_imagenet_to_beans_resnet18_seed42.json

{
  "num_params": 1539,
  "accuracy": 89.0625
}{
  "num_params": 1539,
  "accuracy": 89.84375,
  "beta": 0.75,
  "coeff": 0.5,
  "best_val_acc": 96.99248120300751,
  "val_acc_list": [
    91.72932330827068,
    91.72932330827068,
    92.4812030075188,
    93.98496240601504,
    93.98496240601504,
    96.99248120300751,
    95.48872180451127,
    96.99248120300751,
    96.99248120300751,
    95.48872180451127,
    96.99248120300751,
    93.23308270676692,
    95.48872180451127,
    93.98496240601504,
    93.23308270676692,
    93.98496240601504,
    93.98496240601504,
    90.97744360902256,
    93.98496240601504,
    91.72932330827068,
    93.23308270676692,
    94.73684210526316,
    93.98496240601504,
    92.4812030075188,
    93.23308270676692,
    93.23308270676692,
    91.72932330827068,
    91.72932330827068,
    90.97744360902256,
    92.4812030075188,
    33.08270676691729
  ]
}

In [15]:
!python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800

2026-09-25 19:58:14.280 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed43.log
2026-09-25 19:58:14.284 | INFO     | __main__:main:58 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
2026-09-25 19:58:15.428 | INFO     | __main__:main:85 - Testing baseline...
2026-09-25 19:58:15.442 | INFO     | __main__:main:88 - Number of trainable parameters: 1539
2026-09-25 19:58:15.443 | INFO     | __main__:main:89 - Starting transfer learning...
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: 

In [16]:
!python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds beans \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 800

2026-09-25 20:12:38.122 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_beans_resnet18_seed44.log
2026-09-25 20:12:38.125 | INFO     | __main__:main:58 - Running on cuda
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
2026-09-25 20:12:39.288 | INFO     | __main__:main:85 - Testing baseline...
2026-09-25 20:12:39.303 | INFO     | __main__:main:88 - Number of trainable parameters: 1539
2026-09-25 20:12:39.303 | INFO     | __main__:main:89 - Starting transfer learning...
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: 

In [17]:
import json
import numpy as np

seeds = [42, 43, 44]

for method in ["base", "ct"]:
    accuracies = []
    betas = []

    for seed in seeds:
        path = f"results/{method}_imagenet_to_beans_resnet18_seed{seed}.json"

        with open(path) as f:
            result = json.load(f)

        accuracies.append(result["accuracy"])

        if "beta" in result:
            betas.append(result["beta"])

    print(f"\n{method.upper()}")
    print("Individual accuracies:", accuracies)
    print(f"Mean accuracy: {np.mean(accuracies):.2f}%")
    print(f"Std: {np.std(accuracies):.2f}")

    if betas:
        print("Selected betas:", betas)
        print(f"Mean beta: {np.mean(betas):.2f}")


BASE
Individual accuracies: [89.0625, 91.40625, 87.5]
Mean accuracy: 89.32%
Std: 1.61

CT
Individual accuracies: [89.84375, 89.84375, 89.0625]
Mean accuracy: 89.58%
Std: 0.37
Selected betas: [0.75, 0.77, 0.76]
Mean beta: 0.76
